In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

# Load PCA-reduced data from classical baseline notebook
X_train_pca = np.load('../../data/X_train_pca.npy')
X_test_pca = np.load('../../data/X_test_pca.npy')
y_train = np.load('../../data/y_train.npy')
y_test = np.load('../../data/y_test.npy')

print(f"X_train: {X_train_pca.shape}, X_test: {X_test_pca.shape}")
print(f"y_train fire rate: {y_train.mean():.2%}, y_test fire rate: {y_test.mean():.2%}")

X_train: (7786, 6), X_test: (2594, 6)
y_train fire rate: 8.43%, y_test fire rate: 8.64%


In [2]:
# === Subsample training data for quantum model ===
# Quantum models are slow to train. We need a manageable subset.
# Strategy: balanced subsample (equal fire and no-fire examples)

from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_train_q, y_train_q = rus.fit_resample(X_train_pca, y_train)

print(f"Original training set: {X_train_pca.shape[0]} rows ({y_train.mean():.2%} fire)")
print(f"Balanced quantum training set: {X_train_q.shape[0]} rows ({y_train_q.mean():.2%} fire)")
print(f"  No fire: {(y_train_q == 0).sum()}, Fire: {(y_train_q == 1).sum()}")

# Further subsample if needed (VQC training is slow)
# Start with 400 samples (200 per class), increase later if time permits
QUANTUM_TRAIN_SIZE = 400
if len(X_train_q) > QUANTUM_TRAIN_SIZE:
    np.random.seed(42)
    idx = np.random.choice(len(X_train_q), QUANTUM_TRAIN_SIZE, replace=False)
    X_train_q = X_train_q[idx]
    y_train_q = y_train_q[idx]
    print(f"Subsampled to {QUANTUM_TRAIN_SIZE} for training speed")
    print(f"  No fire: {(y_train_q == 0).sum()}, Fire: {(y_train_q == 1).sum()}")

# Also subsample test set for faster evaluation during training
QUANTUM_TEST_SIZE = 500
np.random.seed(42)
test_idx = np.random.choice(len(X_test_pca), QUANTUM_TEST_SIZE, replace=False)
X_test_q = X_test_pca[test_idx]
y_test_q = y_test[test_idx]
print(f"\nQuantum test subset: {X_test_q.shape[0]} rows ({y_test_q.mean():.2%} fire)")

Original training set: 7786 rows (8.43% fire)
Balanced quantum training set: 1312 rows (50.00% fire)
  No fire: 656, Fire: 656
Subsampled to 400 for training speed
  No fire: 210, Fire: 190

Quantum test subset: 500 rows (8.20% fire)


In [5]:
# === Build the Variational Quantum Classifier ===
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA
from qiskit_aer import AerSimulator

# Number of qubits = number of features = 6
N_QUBITS = 6

# STEP 1: Feature Map
# This encodes our 6 PCA features into quantum states.
# ZZFeatureMap creates entanglement between pairs of features,
# which lets the circuit capture feature interactions.
feature_map = ZZFeatureMap(feature_dimension=N_QUBITS, reps=2, entanglement='linear')

# STEP 2: Ansatz (trainable circuit)
# RealAmplitudes is a parameterized circuit whose rotation angles
# get optimized during training. reps=2 means 2 layers of rotations.
ansatz = RealAmplitudes(num_qubits=N_QUBITS, reps=2, entanglement='linear')

# STEP 3: Optimizer
# COBYLA is a classical optimizer that adjusts the ansatz parameters
# to minimize classification error. maxiter controls training duration.
optimizer = COBYLA(maxiter=100)

# STEP 4: Build VQC
vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    num_qubits=N_QUBITS,
)

print("=== Quantum Circuit Summary ===")
circuit = feature_map.compose(ansatz)
print(f"Qubits: {N_QUBITS}")
print(f"Feature map: ZZFeatureMap (reps=2, linear entanglement)")
print(f"Ansatz: RealAmplitudes (reps=2, linear entanglement)")
print(f"Total parameters: {ansatz.num_parameters}")
print(f"Circuit depth: {circuit.depth()}")
print(f"Optimizer: COBYLA (maxiter=100)")
print(f"\nTraining samples: {len(X_train_q)}")

# Show the circuit structure
print(f"\n=== Circuit Diagram ===")
print(circuit.draw(output='text', fold=120))

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


=== Quantum Circuit Summary ===
Qubits: 6
Feature map: ZZFeatureMap (reps=2, linear entanglement)
Ansatz: RealAmplitudes (reps=2, linear entanglement)
Total parameters: 18
Circuit depth: 2
Optimizer: COBYLA (maxiter=100)

Training samples: 400

=== Circuit Diagram ===
     ┌──────────────────────────────────────────────┐»
q_0: ┤0                                             ├»
     │                                              │»
q_1: ┤1                                             ├»
     │                                              │»
q_2: ┤2                                             ├»
     │  ZZFeatureMap(x[0],x[1],x[2],x[3],x[4],x[5]) │»
q_3: ┤3                                             ├»
     │                                              │»
q_4: ┤4                                             ├»
     │                                              │»
q_5: ┤5                                             ├»
     └──────────────────────────────────────────────┘»
«     ┌─────────

In [6]:
# === Train the VQC ===
import time

print("Training VQC... (this may take 5-15 minutes)")
print("Each iteration runs the quantum circuit on the simulator for every training sample.")
start_time = time.time()

# Track training progress
iteration_count = [0]
def callback(weights, obj_func_eval):
    iteration_count[0] += 1
    if iteration_count[0] % 10 == 0:
        elapsed = time.time() - start_time
        print(f"  Iteration {iteration_count[0]}: loss={obj_func_eval:.4f}, time={elapsed:.0f}s")

vqc.fit(X_train_q, y_train_q)

train_time = time.time() - start_time
print(f"\nTraining complete in {train_time:.1f} seconds")

Training VQC... (this may take 5-15 minutes)
Each iteration runs the quantum circuit on the simulator for every training sample.

Training complete in 96.9 seconds


In [7]:
# === Evaluate Quantum Model ===

# Predict on full test set (not subsampled)
print("Predicting on full test set...")
y_pred_q = vqc.predict(X_test_pca)

# Evaluation helper
def evaluate_model(name, y_true, y_pred, y_prob=None):
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred):.4f}")
    if y_prob is not None:
        print(f"AUC-ROC:   {roc_auc_score(y_true, y_prob):.4f}")
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"  TN={cm[0][0]:>4d}  FP={cm[0][1]:>4d}")
    print(f"  FN={cm[1][0]:>4d}  TP={cm[1][1]:>4d}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['No Fire', 'Fire'])}")

evaluate_model("Quantum VQC (6 PCA features)", y_test, y_pred_q)

# === Resource requirements (for submission) ===
print(f"\n{'='*50}")
print(f"  Quantum Resource Requirements")
print(f"{'='*50}")
print(f"Qubits: {N_QUBITS}")
print(f"Circuit depth: {circuit.depth()}")
print(f"Trainable parameters: {ansatz.num_parameters}")
print(f"Feature map gates: {feature_map.decompose().count_ops()}")
print(f"Ansatz gates: {ansatz.decompose().count_ops()}")
print(f"Training samples: {len(X_train_q)}")
print(f"Training time: {train_time:.1f} seconds")
print(f"Simulator: Qiskit Aer (statevector)")

Predicting on full test set...

  Quantum VQC (6 PCA features)
Accuracy:  0.5251
Precision: 0.0968
Recall:    0.5402
F1 Score:  0.1642

Confusion Matrix:
  TN=1241  FP=1129
  FN= 103  TP= 121

              precision    recall  f1-score   support

     No Fire       0.92      0.52      0.67      2370
        Fire       0.10      0.54      0.16       224

    accuracy                           0.53      2594
   macro avg       0.51      0.53      0.42      2594
weighted avg       0.85      0.53      0.62      2594


  Quantum Resource Requirements
Qubits: 6
Circuit depth: 2
Trainable parameters: 18
Feature map gates: OrderedDict([('p', 22), ('cx', 20), ('h', 12)])
Ansatz gates: OrderedDict([('ry', 18), ('cx', 10)])
Training samples: 400
Training time: 96.9 seconds
Simulator: Qiskit Aer (statevector)


In [8]:
# === Improved VQC: more iterations, more data ===
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA
import time

# Use full balanced dataset (1312 samples)
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_train_q2, y_train_q2 = rus.fit_resample(X_train_pca, y_train)
print(f"Training set: {X_train_q2.shape[0]} samples ({y_train_q2.mean():.0%} fire)")

N_QUBITS = 6
feature_map = ZZFeatureMap(feature_dimension=N_QUBITS, reps=2, entanglement='linear')
ansatz = RealAmplitudes(num_qubits=N_QUBITS, reps=3, entanglement='linear')  # increased to 3 reps
optimizer = COBYLA(maxiter=200)  # doubled iterations

vqc2 = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    num_qubits=N_QUBITS,
)

print(f"Ansatz parameters: {ansatz.num_parameters} (was 18, now more with reps=3)")
print(f"Optimizer iterations: 200")
print(f"\nTraining... (may take 5-10 minutes)")

start = time.time()
vqc2.fit(X_train_q2, y_train_q2)
train_time2 = time.time() - start
print(f"Training complete in {train_time2:.1f} seconds")

# Evaluate
y_pred_q2 = vqc2.predict(X_test_pca)
evaluate_model("Quantum VQC v2 (more data + iterations)", y_test, y_pred_q2)

# Resource requirements
circuit2 = feature_map.compose(ansatz)
print(f"\n=== Updated Resource Requirements ===")
print(f"Qubits: {N_QUBITS}")
print(f"Circuit depth: {circuit2.depth()}")
print(f"Trainable parameters: {ansatz.num_parameters}")
print(f"Training samples: {len(X_train_q2)}")
print(f"Training time: {train_time2:.1f} seconds")

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


Training set: 1312 samples (50% fire)
Ansatz parameters: 24 (was 18, now more with reps=3)
Optimizer iterations: 200

Training... (may take 5-10 minutes)
Training complete in 670.7 seconds

  Quantum VQC v2 (more data + iterations)
Accuracy:  0.4156
Precision: 0.0880
Recall:    0.6161
F1 Score:  0.1540

Confusion Matrix:
  TN= 940  FP=1430
  FN=  86  TP= 138

              precision    recall  f1-score   support

     No Fire       0.92      0.40      0.55      2370
        Fire       0.09      0.62      0.15       224

    accuracy                           0.42      2594
   macro avg       0.50      0.51      0.35      2594
weighted avg       0.84      0.42      0.52      2594


=== Updated Resource Requirements ===
Qubits: 6
Circuit depth: 2
Trainable parameters: 24
Training samples: 1312
Training time: 670.7 seconds


In [9]:
# === Approach 2: Quantum Kernel SVM ===
# Instead of VQC, we use a quantum circuit to compute a kernel (similarity matrix)
# and feed it into a classical SVM. This is often more stable.

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from sklearn.svm import SVC
import time

# Use a smaller but more realistic training set
# Subsample to 500 total with natural class ratio (~8.5% fire)
np.random.seed(42)
n_total = 500
fire_idx = np.where(y_train == 1)[0]
nofire_idx = np.where(y_train == 0)[0]

# Keep natural ratio: ~43 fire, ~457 no-fire
n_fire = min(43, len(fire_idx))
n_nofire = n_total - n_fire
fire_sample = np.random.choice(fire_idx, n_fire, replace=False)
nofire_sample = np.random.choice(nofire_idx, n_nofire, replace=False)
sample_idx = np.concatenate([fire_sample, nofire_sample])
np.random.shuffle(sample_idx)

X_train_kern = X_train_pca[sample_idx]
y_train_kern = y_train[sample_idx]
print(f"Kernel training set: {len(X_train_kern)} samples, {y_train_kern.mean():.2%} fire rate")

# Build quantum kernel
N_QUBITS = 6
feature_map_kern = ZZFeatureMap(feature_dimension=N_QUBITS, reps=2, entanglement='linear')
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map_kern)

# Compute kernel matrix (this is the quantum step)
print("Computing quantum kernel matrix... (this may take a few minutes)")
start = time.time()
kernel_matrix_train = quantum_kernel.evaluate(X_train_kern)
kernel_time = time.time() - start
print(f"Kernel matrix computed in {kernel_time:.1f}s, shape: {kernel_matrix_train.shape}")

# Train classical SVM on quantum kernel
svm = SVC(kernel='precomputed', class_weight='balanced', probability=True, random_state=42)
svm.fit(kernel_matrix_train, y_train_kern)
print("SVM trained on quantum kernel.")

# Evaluate: compute kernel between test and train points
print("Computing test kernel matrix...")
start = time.time()
kernel_matrix_test = quantum_kernel.evaluate(X_test_pca, X_train_kern)
test_kernel_time = time.time() - start
print(f"Test kernel computed in {test_kernel_time:.1f}s")

y_pred_kern = svm.predict(kernel_matrix_test)
y_prob_kern = svm.predict_proba(kernel_matrix_test)[:, 1]

evaluate_model("Quantum Kernel SVM (6 PCA features)", y_test, y_pred_kern, y_prob_kern)

print(f"\n=== Quantum Kernel Resource Requirements ===")
print(f"Qubits: {N_QUBITS}")
print(f"Feature map: ZZFeatureMap (reps=2, linear)")
print(f"Kernel matrix train: {kernel_matrix_train.shape}")
print(f"Kernel computation time: {kernel_time:.1f}s (train) + {test_kernel_time:.1f}s (test)")
print(f"Classical SVM on top of quantum kernel")

Kernel training set: 500 samples, 8.60% fire rate
Computing quantum kernel matrix... (this may take a few minutes)
Kernel matrix computed in 346.8s, shape: (500, 500)
SVM trained on quantum kernel.
Computing test kernel matrix...
Test kernel computed in 3698.3s

  Quantum Kernel SVM (6 PCA features)
Accuracy:  0.9082
Precision: 0.1111
Recall:    0.0089
F1 Score:  0.0165
AUC-ROC:   0.4865

Confusion Matrix:
  TN=2354  FP=  16
  FN= 222  TP=   2

              precision    recall  f1-score   support

     No Fire       0.91      0.99      0.95      2370
        Fire       0.11      0.01      0.02       224

    accuracy                           0.91      2594
   macro avg       0.51      0.50      0.48      2594
weighted avg       0.84      0.91      0.87      2594


=== Quantum Kernel Resource Requirements ===
Qubits: 6
Feature map: ZZFeatureMap (reps=2, linear)
Kernel matrix train: (500, 500)
Kernel computation time: 346.8s (train) + 3698.3s (test)
Classical SVM on top of quantum kern

In [10]:
# === VQC with threshold tuning ===
# Our VQC v1 was the best so far. Let's retrain it and tune the threshold.
# The problem was: trained on 50/50 data, tested on 91.4/8.6 data.
# Solution: instead of predicting class directly, get probabilities and pick threshold.

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA
import time

# Balanced training set (400 samples, 50/50)
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_train_bal, y_train_bal = rus.fit_resample(X_train_pca, y_train)
np.random.seed(42)
idx = np.random.choice(len(X_train_bal), 400, replace=False)
X_train_bal = X_train_bal[idx]
y_train_bal = y_train_bal[idx]

N_QUBITS = 6
feature_map = ZZFeatureMap(feature_dimension=N_QUBITS, reps=2, entanglement='linear')
ansatz = RealAmplitudes(num_qubits=N_QUBITS, reps=2, entanglement='linear')
optimizer = COBYLA(maxiter=150)

vqc3 = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    num_qubits=N_QUBITS,
)

print(f"Training VQC v3 (400 samples, 150 iterations)...")
start = time.time()
vqc3.fit(X_train_bal, y_train_bal)
t = time.time() - start
print(f"Done in {t:.0f}s")

# Get raw scores for threshold tuning
# VQC predict gives class labels, but we need the underlying circuit outputs
# Use predict on test set to get labels, then we'll work with the scores
y_pred_v3 = vqc3.predict(X_test_pca)

# Check what the raw predictions look like
unique, counts = np.unique(y_pred_v3, return_counts=True)
print(f"\nRaw predictions: {dict(zip(unique.astype(int), counts))}")

# Try the default predictions first
evaluate_model("VQC v3 (default threshold)", y_test, y_pred_v3)

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


Training VQC v3 (400 samples, 150 iterations)...
Done in 160s

Raw predictions: {np.int64(0): np.int64(1411), np.int64(1): np.int64(1183)}

  VQC v3 (default threshold)
Accuracy:  0.5424
Precision: 0.0930
Recall:    0.4911
F1 Score:  0.1564

Confusion Matrix:
  TN=1297  FP=1073
  FN= 114  TP= 110

              precision    recall  f1-score   support

     No Fire       0.92      0.55      0.69      2370
        Fire       0.09      0.49      0.16       224

    accuracy                           0.54      2594
   macro avg       0.51      0.52      0.42      2594
weighted avg       0.85      0.54      0.64      2594



In [11]:
# === VQC with SPSA optimizer (often better for quantum) ===
from qiskit_algorithms.optimizers import SPSA

feature_map4 = ZZFeatureMap(feature_dimension=N_QUBITS, reps=2, entanglement='linear')
ansatz4 = RealAmplitudes(num_qubits=N_QUBITS, reps=2, entanglement='linear')
optimizer4 = SPSA(maxiter=150)

vqc4 = VQC(
    feature_map=feature_map4,
    ansatz=ansatz4,
    optimizer=optimizer4,
    num_qubits=N_QUBITS,
)

print(f"Training VQC v4 (SPSA optimizer, 400 samples, 150 iterations)...")
start = time.time()
vqc4.fit(X_train_bal, y_train_bal)
t4 = time.time() - start
print(f"Done in {t4:.0f}s")

y_pred_v4 = vqc4.predict(X_test_pca)
evaluate_model("VQC v4 (SPSA optimizer)", y_test, y_pred_v4)

# === Final comparison table ===
print(f"\n{'='*65}")
print(f"  FULL COMPARISON (all models on 2021 test set)")
print(f"{'='*65}")
print(f"{'Model':<40s} {'Acc':>5s} {'Prec':>5s} {'Rec':>5s} {'F1':>5s}")
print(f"{'-'*65}")

results = [
    ("RF (43 features)", 0.8601, 0.3505, 0.7277, 0.4731),
    ("XGB (43 features)", 0.8890, 0.4116, 0.6652, 0.5085),
    ("RF (6 PCA features)", 0.8524, 0.3268, 0.6696, 0.4392),
    ("XGB (6 PCA features)", 0.8466, 0.3125, 0.6473, 0.4215),
]

for name, acc, prec, rec, f1 in results:
    print(f"{name:<40s} {acc:>5.3f} {prec:>5.3f} {rec:>5.3f} {f1:>5.3f}")

# Add quantum results
for name, y_p in [("VQC v1 (100 iter)", y_pred_q), 
                   ("VQC v3 (150 iter)", y_pred_v3),
                   ("VQC v4 (SPSA)", y_pred_v4)]:
    acc = accuracy_score(y_test, y_p)
    prec = precision_score(y_test, y_p, zero_division=0)
    rec = recall_score(y_test, y_p)
    f1 = f1_score(y_test, y_p)
    print(f"{name:<40s} {acc:>5.3f} {prec:>5.3f} {rec:>5.3f} {f1:>5.3f}")

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


Training VQC v4 (SPSA optimizer, 400 samples, 150 iterations)...
Done in 356s

  VQC v4 (SPSA optimizer)
Accuracy:  0.4811
Precision: 0.0763
Recall:    0.4509
F1 Score:  0.1305

Confusion Matrix:
  TN=1147  FP=1223
  FN= 123  TP= 101

              precision    recall  f1-score   support

     No Fire       0.90      0.48      0.63      2370
        Fire       0.08      0.45      0.13       224

    accuracy                           0.48      2594
   macro avg       0.49      0.47      0.38      2594
weighted avg       0.83      0.48      0.59      2594


  FULL COMPARISON (all models on 2021 test set)
Model                                      Acc  Prec   Rec    F1
-----------------------------------------------------------------
RF (43 features)                         0.860 0.350 0.728 0.473
XGB (43 features)                        0.889 0.412 0.665 0.508
RF (6 PCA features)                      0.852 0.327 0.670 0.439
XGB (6 PCA features)                     0.847 0.312 0.647 0.42

In [12]:
# === 10-Qubit VQC ===
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA
from imblearn.under_sampling import RandomUnderSampler
import time

# Load 10-component PCA data
X_train_10 = np.load('../../data/X_train_pca10.npy')
X_test_10 = np.load('../../data/X_test_pca10.npy')

# Balanced subsample
rus = RandomUnderSampler(random_state=42)
X_bal, y_bal = rus.fit_resample(X_train_10, y_train)
np.random.seed(42)
idx = np.random.choice(len(X_bal), 400, replace=False)
X_bal = X_bal[idx]
y_bal = y_bal[idx]
print(f"Training: {X_bal.shape}, fire rate: {y_bal.mean():.0%}")

N_QUBITS = 10
feature_map = ZZFeatureMap(feature_dimension=N_QUBITS, reps=2, entanglement='linear')
ansatz = RealAmplitudes(num_qubits=N_QUBITS, reps=2, entanglement='linear')
optimizer = COBYLA(maxiter=150)

vqc10 = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    num_qubits=N_QUBITS,
)

circuit10 = feature_map.compose(ansatz)
print(f"Qubits: {N_QUBITS}, Parameters: {ansatz.num_parameters}, Depth: {circuit10.depth()}")

print(f"\nTraining 10-qubit VQC...")
start = time.time()
vqc10.fit(X_bal, y_bal)
t = time.time() - start
print(f"Done in {t:.0f}s")

y_pred_10q = vqc10.predict(X_test_10)
evaluate_model("Quantum VQC 10-qubit (COBYLA, 150 iter)", y_test, y_pred_10q)

print(f"\n=== Resource Requirements ===")
print(f"Qubits: {N_QUBITS}")
print(f"Circuit depth: {circuit10.depth()}")
print(f"Trainable parameters: {ansatz.num_parameters}")
print(f"Training time: {t:.1f}s")

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


Training: (400, 10), fire rate: 48%
Qubits: 10, Parameters: 30, Depth: 2

Training 10-qubit VQC...
Done in 1470s

  Quantum VQC 10-qubit (COBYLA, 150 iter)
Accuracy:  0.5424
Precision: 0.0971
Recall:    0.5179
F1 Score:  0.1635

Confusion Matrix:
  TN=1291  FP=1079
  FN= 108  TP= 116

              precision    recall  f1-score   support

     No Fire       0.92      0.54      0.69      2370
        Fire       0.10      0.52      0.16       224

    accuracy                           0.54      2594
   macro avg       0.51      0.53      0.42      2594
weighted avg       0.85      0.54      0.64      2594


=== Resource Requirements ===
Qubits: 10
Circuit depth: 2
Trainable parameters: 30
Training time: 1470.5s


In [13]:
# === Best shot: 10-qubit VQC with top features, more data, more iterations ===
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA
from imblearn.under_sampling import RandomUnderSampler
import time

# Load top-10 features (real features, not PCA)
X_train_top10 = np.load('../../data/X_train_top10.npy')
X_test_top10 = np.load('../../data/X_test_top10.npy')

# Use FULL balanced dataset this time
rus = RandomUnderSampler(random_state=42)
X_bal, y_bal = rus.fit_resample(X_train_top10, y_train)
print(f"Full balanced training set: {X_bal.shape}, fire rate: {y_bal.mean():.0%}")

N_QUBITS = 10
feature_map = ZZFeatureMap(feature_dimension=N_QUBITS, reps=2, entanglement='linear')
ansatz = RealAmplitudes(num_qubits=N_QUBITS, reps=2, entanglement='linear')
optimizer = COBYLA(maxiter=300)

vqc_best = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    num_qubits=N_QUBITS,
)

circuit = feature_map.compose(ansatz)
print(f"Qubits: {N_QUBITS}, Parameters: {ansatz.num_parameters}")
print(f"Training samples: {len(X_bal)}, Iterations: 300")
print(f"\nThis will take a while. Go grab a coffee.")

start = time.time()
vqc_best.fit(X_bal, y_bal)
t = time.time() - start
print(f"Done in {t/60:.1f} minutes")

y_pred_best = vqc_best.predict(X_test_top10)
evaluate_model("VQC Best (top-10 features, full data, 300 iter)", y_test, y_pred_best)

print(f"\n=== Resource Requirements ===")
print(f"Qubits: {N_QUBITS}")
print(f"Circuit depth: {circuit.depth()}")
print(f"Trainable parameters: {ansatz.num_parameters}")
print(f"Feature map: ZZFeatureMap (reps=2, linear)")
print(f"Ansatz: RealAmplitudes (reps=2, linear)")
print(f"Training samples: {len(X_bal)}")
print(f"Optimizer: COBYLA (maxiter=300)")
print(f"Training time: {t/60:.1f} minutes")
print(f"Simulator: Qiskit Aer (statevector)")

# Print which features we used (important for the report)
with open('../../data/top10_features.txt', 'r') as f:
    features = f.read().strip().split('\n')
print(f"\nFeatures used:")
for i, feat in enumerate(features):
    print(f"  q{i}: {feat}")

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


Full balanced training set: (1312, 10), fire rate: 50%
Qubits: 10, Parameters: 30
Training samples: 1312, Iterations: 300

This will take a while. Go grab a coffee.
Done in 158.2 minutes

  VQC Best (top-10 features, full data, 300 iter)
Accuracy:  0.7930
Precision: 0.2030
Recall:    0.4777
F1 Score:  0.2850

Confusion Matrix:
  TN=1950  FP= 420
  FN= 117  TP= 107

              precision    recall  f1-score   support

     No Fire       0.94      0.82      0.88      2370
        Fire       0.20      0.48      0.28       224

    accuracy                           0.79      2594
   macro avg       0.57      0.65      0.58      2594
weighted avg       0.88      0.79      0.83      2594


=== Resource Requirements ===
Qubits: 10
Circuit depth: 2
Trainable parameters: 30
Feature map: ZZFeatureMap (reps=2, linear)
Ansatz: RealAmplitudes (reps=2, linear)
Training samples: 1312
Optimizer: COBYLA (maxiter=300)
Training time: 158.2 minutes
Simulator: Qiskit Aer (statevector)

Features used:
  

In [15]:
# === Experiment: 3 variations to find best quantum model ===
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes, EfficientSU2
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA, SPSA, L_BFGS_B
from imblearn.under_sampling import RandomUnderSampler
import time

# Load top-10 features
X_train_top10 = np.load('../../data/X_train_top10.npy')
X_test_top10 = np.load('../../data/X_test_top10.npy')

# Full balanced training set
rus = RandomUnderSampler(random_state=42)
X_bal_10, y_bal_10 = rus.fit_resample(X_train_top10, y_train)

results = []

# === Experiment 1: 10 qubits, L_BFGS_B optimizer ===
print("="*60)
print("  Experiment 1: 10q, top-10 features, L_BFGS_B, 300 iter")
print("="*60)

fm1 = ZZFeatureMap(feature_dimension=10, reps=2, entanglement='linear')
an1 = RealAmplitudes(num_qubits=10, reps=2, entanglement='linear')
opt1 = L_BFGS_B(maxiter=300)

vqc_e1 = VQC(feature_map=fm1, ansatz=an1, optimizer=opt1, num_qubits=10)
start = time.time()
vqc_e1.fit(X_bal_10, y_bal_10)
t1 = time.time() - start
print(f"Done in {t1/60:.1f} min")

pred1 = vqc_e1.predict(X_test_top10)
evaluate_model("Exp1: 10q L_BFGS_B", y_test, pred1)
results.append(("10q L_BFGS_B top10", pred1, t1))

# === Experiment 2: 10 qubits, full entanglement, COBYLA ===
print("\n" + "="*60)
print("  Experiment 2: 10q, top-10 features, full entanglement, 300 iter")
print("="*60)

fm2 = ZZFeatureMap(feature_dimension=10, reps=2, entanglement='full')
an2 = RealAmplitudes(num_qubits=10, reps=2, entanglement='full')
opt2 = COBYLA(maxiter=300)

vqc_e2 = VQC(feature_map=fm2, ansatz=an2, optimizer=opt2, num_qubits=10)
start = time.time()
vqc_e2.fit(X_bal_10, y_bal_10)
t2 = time.time() - start
print(f"Done in {t2/60:.1f} min")

pred2 = vqc_e2.predict(X_test_top10)
evaluate_model("Exp2: 10q full entanglement", y_test, pred2)
results.append(("10q full entangle COBYLA", pred2, t2))

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


  Experiment 1: 10q, top-10 features, L_BFGS_B, 300 iter


KeyboardInterrupt: 

In [ ]:
# === Experiment 3: 12 qubits with top-12 features ===
# Build top-12 feature set first
import pandas as pd
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

df = pd.read_csv('../../data/feature_matrix_clean.csv')
feature_cols = [col for col in df.columns if col not in ['zip', 'Year', 'had_fire']]
train = df[df['Year'].isin([2018, 2019, 2020])]
test = df[df['Year'] == 2021]
X_tr = train[feature_cols]
y_tr = train['had_fire']
X_te = test[feature_cols]

scaler = StandardScaler()
X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr), columns=feature_cols, index=X_tr.index)
X_te_s = pd.DataFrame(scaler.transform(X_te), columns=feature_cols, index=X_te.index)

xgb_temp = XGBClassifier(n_estimators=200, max_depth=6, random_state=42, eval_metric='logloss')
xgb_temp.fit(X_tr_s, y_tr)
importance = pd.DataFrame({'feature': feature_cols, 'importance': xgb_temp.feature_importances_})
importance = importance.sort_values('importance', ascending=False)

top12 = importance.head(12)['feature'].tolist()
print("Top 12 features:")
for f in top12:
    print(f"  {f}")

X_train_top12 = X_tr_s[top12].values
X_test_top12 = X_te_s[top12].values

# Full balanced set
rus = RandomUnderSampler(random_state=42)
X_bal_12, y_bal_12 = rus.fit_resample(X_train_top12, y_tr.values)

print(f"\nTraining: {X_bal_12.shape}, fire rate: {y_bal_12.mean():.0%}")

print("\n" + "="*60)
print("  Experiment 3: 12q, top-12 features, COBYLA, 400 iter")
print("="*60)

fm3 = ZZFeatureMap(feature_dimension=12, reps=2, entanglement='linear')
an3 = RealAmplitudes(num_qubits=12, reps=2, entanglement='linear')
opt3 = COBYLA(maxiter=400)

vqc_e3 = VQC(feature_map=fm3, ansatz=an3, optimizer=opt3, num_qubits=12)
start = time.time()
vqc_e3.fit(X_bal_12, y_bal_12)
t3 = time.time() - start
print(f"Done in {t3/60:.1f} min")

pred3 = vqc_e3.predict(X_test_top12)
evaluate_model("Exp3: 12q top-12 COBYLA 400iter", y_test, pred3)

# === Summary of all experiments ===
print(f"\n{'='*75}")
print(f"  ALL QUANTUM EXPERIMENTS")
print(f"{'='*75}")
print(f"{'Model':<50s} {'Acc':>5s} {'Prec':>5s} {'Rec':>5s} {'F1':>5s} {'Time':>6s}")
print(f"{'-'*75}")

all_quantum = [
    ("VQC 6q PCA COBYLA 100i 400s", 0.525, 0.097, 0.540, 0.164, "1.6m"),
    ("VQC 10q top10 COBYLA 300i 1312s", 0.793, 0.203, 0.478, 0.285, "158m"),
]

for name, pred, t in results:
    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    print(f"{name:<50s} {acc:>5.3f} {prec:>5.3f} {rec:>5.3f} {f1:>5.3f} {t/60:>5.1f}m")

acc3 = accuracy_score(y_test, pred3)
prec3 = precision_score(y_test, pred3, zero_division=0)
rec3 = recall_score(y_test, pred3)
f13 = f1_score(y_test, pred3)
print(f"{'12q top12 COBYLA 400i':<50s} {acc3:>5.3f} {prec3:>5.3f} {rec3:>5.3f} {f13:>5.3f} {t3/60:>5.1f}m")